# 🧠 Toy Neural Network — From Scratch with NumPy

A ground-up implementation of a feedforward neural network using **only NumPy** — no PyTorch, no Keras.  
Building every piece by hand is the best way to truly understand deep learning.

## Learning Objectives
By the end of this notebook you will be able to:
1. Generate and visualise a non-linearly-separable 2-D dataset
2. Implement **sigmoid**, **ReLU**, and **tanh** activation functions and their derivatives
3. Build a flexible multi-layer network with **Xavier weight initialisation**
4. Code **forward propagation** layer-by-layer
5. Derive and code **binary cross-entropy loss**
6. Implement **backpropagation** using the chain rule
7. Train the network and **plot the loss curve**
8. Visualise the **decision boundary** learned by the network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

## §1 — Generate & Visualise the Dataset

We use **`make_moons`** — two interleaving half-circles that a straight line *cannot* separate.  
This forces the network to learn a **non-linear** decision boundary.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Generate dataset ─────────────────────────────────────────
X, y = make_moons(n_samples=600, noise=0.25, random_state=42)
y = y.reshape(-1, 1).astype(float)

# Standardise features (zero mean, unit variance)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train / test split  (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} samples   Test: {X_test.shape[0]} samples")
print(f"Class balance  — train: {y_train.mean():.2f}  test: {y_test.mean():.2f}")

# ── Visualise ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#4878d0', '#ee854a']
for cls, label in [(0, 'Class 0'), (1, 'Class 1')]:
    mask = (y_train.ravel() == cls)
    ax.scatter(X_train[mask, 0], X_train[mask, 1],
               c=colors[cls], label=label, alpha=0.7, edgecolors='white', linewidths=0.4)
ax.set_title('make_moons — training data')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout(); plt.show()

## §2 — Activation Functions & Derivatives

Activation functions introduce **non-linearity**, allowing the network to learn complex patterns.

| Function | Formula | Derivative | Use-case |
|---|---|---|---|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | Output layer (binary) |
| Tanh | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - \tanh^2(z)$ | Hidden layers |
| ReLU | $\max(0, z)$ | $\mathbb{1}[z > 0]$ | Hidden layers (modern) |

In [ ]:
# ── Activation functions ────────────────────────────────────
def sigmoid(z):      return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def sigmoid_d(z):    s = sigmoid(z); return s * (1 - s)
def tanh_act(z):     return np.tanh(z)
def tanh_d(z):       return 1 - np.tanh(z)**2
def relu(z):         return np.maximum(0, z)
def relu_d(z):       return (z > 0).astype(float)

ACTIVATIONS = {
    'sigmoid': (sigmoid, sigmoid_d),
    'tanh':    (tanh_act, tanh_d),
    'relu':    (relu, relu_d),
}

# ── Plot all three ───────────────────────────────────────────
z = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for ax, (name, (fn, dfn)) in zip(axes, ACTIVATIONS.items()):
    ax.plot(z, fn(z),  lw=2.5, label=f'{name}(z)')
    ax.plot(z, dfn(z), lw=2,   ls='--', label=f"derivative")
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(name.capitalize()); ax.legend(fontsize=9)
plt.suptitle('Activation Functions & Their Derivatives', fontweight='bold')
plt.tight_layout(); plt.show()

## §3 — Network Architecture & Weight Initialisation

Our network is a fully-connected MLP with configurable hidden layers.

```
Input (2)  →  Hidden 1 (16, tanh)  →  Hidden 2 (8, tanh)  →  Output (1, sigmoid)
```

**Xavier / Glorot initialisation**: sets the initial weight variance to $\frac{2}{n_{in}+n_{out}}$, preventing gradients from vanishing or exploding in early training.

In [ ]:
def init_params(layer_dims):
    """Xavier-initialised weights and zero biases.
    layer_dims: list of ints, e.g. [2, 16, 8, 1]
    Returns: list of dicts with keys 'W' and 'b'.
    """
    params = []
    for i in range(len(layer_dims) - 1):
        n_in, n_out = layer_dims[i], layer_dims[i + 1]
        scale = np.sqrt(2.0 / (n_in + n_out))   # Xavier
        params.append({
            'W': np.random.randn(n_in, n_out) * scale,
            'b': np.zeros((1, n_out))
        })
    return params

# Default architecture: 2 inputs → 16 → 8 → 1 output
ARCH = [2, 16, 8, 1]
params = init_params(ARCH)

print("Layer shapes (W, b):")
for i, p in enumerate(params):
    print(f"  Layer {i+1}: W={p['W'].shape}  b={p['b'].shape}")

## §4 — Forward Propagation

For each layer $l$:
$$z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}, \qquad A^{[l]} = g^{[l]}(z^{[l]})$$

Hidden layers use **tanh**; the final layer uses **sigmoid** to produce a probability in $[0, 1]$.

In [ ]:
def forward(X, params, hidden_act='tanh'):
    """Run forward pass.
    Returns: output probabilities (n, 1), and cache of (Z, A) per layer.
    """
    act_fn, _ = ACTIVATIONS[hidden_act]
    cache = []
    A = X
    for i, p in enumerate(params):
        Z = A @ p['W'] + p['b']
        # Last layer → sigmoid; hidden layers → chosen activation
        A = sigmoid(Z) if i == len(params) - 1 else act_fn(Z)
        cache.append((Z, A))
    return A, cache   # A is final output ŷ

# Quick sanity check
y_hat, _ = forward(X_train[:5], params)
print("Output probabilities (first 5 samples):", y_hat.ravel().round(4))

## §5 — Loss Function: Binary Cross-Entropy

For binary classification the standard loss is:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\bigl[y_i \log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\bigr]$$

A small constant $\epsilon = 10^{-8}$ is added inside the log to prevent $\log(0)$.

In [ ]:
eps = 1e-8

def bce_loss(y_true, y_hat):
    """Binary cross-entropy loss."""
    n = y_true.shape[0]
    return -np.mean(y_true * np.log(y_hat + eps) + (1 - y_true) * np.log(1 - y_hat + eps))

def accuracy(y_true, y_hat, threshold=0.5):
    preds = (y_hat >= threshold).astype(float)
    return np.mean(preds == y_true)

# Test on random weights
y_hat_test, _ = forward(X_train, params)
print(f"Initial loss:    {bce_loss(y_train, y_hat_test):.4f}")
print(f"Initial accuracy:{accuracy(y_train, y_hat_test):.4f}  (expect ~0.50)")

## §6 — Backpropagation

We propagate the error *backwards* using the **chain rule**.  
For the output layer (sigmoid + BCE the gradient simplifies to):

$$\delta^{[L]} = \hat{y} - y$$

For each hidden layer $l$ (going right → left):

$$\delta^{[l]} = (\delta^{[l+1]} W^{[l+1]\top}) \odot g'^{[l]}(z^{[l]})$$

Weight updates via **gradient descent** with learning rate $\alpha$:

$$W^{[l]} \leftarrow W^{[l]} - \frac{\alpha}{n} A^{[l-1]\top} \delta^{[l]}, \qquad b^{[l]} \leftarrow b^{[l]} - \frac{\alpha}{n}\sum \delta^{[l]}$$

In [ ]:
def backward(X, y_true, params, cache, hidden_act='tanh', lr=0.01):
    """One gradient-descent step.
    Updates params in-place and returns the loss for this batch.
    """
    n = X.shape[0]
    _, act_d = ACTIVATIONS[hidden_act]

    # Build list of activations: A[0]=X, A[1..L]=cache activations
    A_list = [X] + [c[1] for c in cache]
    Z_list = [c[0] for c in cache]
    L = len(params)

    # Output layer delta (BCE + sigmoid shortcut: dL/dz = ŷ − y)
    delta = A_list[-1] - y_true

    for l in range(L - 1, -1, -1):
        dW = (A_list[l].T @ delta) / n
        db = delta.mean(axis=0, keepdims=True)

        if l > 0:
            # Propagate gradient to previous layer
            delta = (delta @ params[l]['W'].T) * act_d(Z_list[l - 1])

        # Update weights
        params[l]['W'] -= lr * dW
        params[l]['b'] -= lr * db

    y_hat = A_list[-1]
    return bce_loss(y_true, y_hat)

## §7 — Train the Network

We now run the full **forward → loss → backward** loop for 2 000 epochs and track loss and accuracy on both train and test sets.

In [ ]:
def train(X_tr, y_tr, X_te, y_te, layer_dims, hidden_act='tanh',
          lr=0.01, epochs=2000, verbose=True):
    """Full training loop. Returns trained params and history dict."""
    np.random.seed(42)
    params = init_params(layer_dims)
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        y_hat_tr, cache = forward(X_tr, params, hidden_act)
        backward(X_tr, y_tr, params, cache, hidden_act, lr)

        y_hat_te, _ = forward(X_te, params, hidden_act)
        history['train_loss'].append(bce_loss(y_tr, y_hat_tr))
        history['test_loss'].append(bce_loss(y_te, y_hat_te))
        history['train_acc'].append(accuracy(y_tr, y_hat_tr))
        history['test_acc'].append(accuracy(y_te, y_hat_te))

        if verbose and epoch % 400 == 0:
            print(f"Epoch {epoch:4d}  train_loss={history['train_loss'][-1]:.4f}"
                  f"  test_acc={history['test_acc'][-1]:.4f}")
    return params, history

# ── Train the default 2-16-8-1 network ──────────────────────
trained_params, history = train(X_train, y_train, X_test, y_test,
                                ARCH, hidden_act='tanh', lr=0.05, epochs=2000)

# ── Loss & accuracy curves ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, 2001)
ax1.plot(epochs_range, history['train_loss'], label='Train loss')
ax1.plot(epochs_range, history['test_loss'],  label='Test loss',  ls='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE loss'); ax1.set_title('Loss Curves')
ax1.legend()

ax2.plot(epochs_range, history['train_acc'], label='Train accuracy')
ax2.plot(epochs_range, history['test_acc'],  label='Test accuracy', ls='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy Curves')
ax2.legend()
plt.tight_layout(); plt.show()

## §8 — Decision Boundary Visualisation

We tile a fine mesh over the 2-D feature space, run each point through the trained network, and colour the background by the predicted class.  
The actual data points are overlaid so we can judge how well the boundary fits.

In [ ]:
def plot_decision_boundary(params, X, y, title='Decision Boundary', ax=None):
    h = 0.03
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z, _ = forward(grid, params)
    Z = Z.reshape(xx.shape)

    _ax = ax or plt.gca()
    _ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.7)
    _ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.5)
    y_flat = y.ravel()
    for cls, marker, label in [(0, 'o', 'Class 0'), (1, 's', 'Class 1')]:
        mask = y_flat == cls
        _ax.scatter(X[mask, 0], X[mask, 1], marker=marker,
                    c=['#4878d0', '#ee854a'][cls], edgecolors='white',
                    linewidths=0.4, s=35, label=label, zorder=3)
    _ax.set_title(title); _ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(trained_params, X_train, y_train, 'Train set', axes[0])
plot_decision_boundary(trained_params, X_test,  y_test,  'Test set',  axes[1])
plt.suptitle('Learned Decision Boundary (2-16-8-1 MLP, tanh)', fontweight='bold')
plt.tight_layout(); plt.show()